<a href="https://colab.research.google.com/github/dfisher3/SeniorDesign/blob/main/Cipro_Mass_Balance_(Revised).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Energy Balance Assumptions

Steps 1 & 2 (Acylation & Enamine Formation): The first stage of this synthesis involves an acyl chloride reacting with an acrylate derivative, followed by an amine exchange. Acyl chloride substitutions are notoriously exothermic due to the high energy of the C-Cl bond compared to the resulting C-C or C-N bonds. Standard Schotten-Baumann-type acylations release between $-70 \text{ to } -100\text{ kJ/mol}$. We will use a calculated estimate of $-80\text{ kJ/mol}$.

Step 4 (S_NAr / Ring Closure with Piperazine): We can accurately estimate this using standard Bond Dissociation Energies (BDEs).<br>Bonds Broken: Aryl C–F bond ($\sim 523\text{ kJ/mol}$) + Piperazine N–H bond ($\sim 390\text{ kJ/mol}$) = $913\text{ kJ/mol}$ required.
<br>Bonds Formed: Aryl C–N bond ($\sim 430\text{ kJ/mol}$) + H–F bond ($\sim 570\text{ kJ/mol}$) = $1000\text{ kJ/mol}$ released.
<br>Estimated $\Delta H_{rxn}$ = $913 - 1000 =$ $-87\text{ kJ/mol}$.

Step 5 (Base-Catalyzed Ester Hydrolysis): The final upstream step uses TBAOH to hydrolyze the ethyl ester. Base-catalyzed saponification is a textbook thermodynamic value. The literature standard for the saponification of an ethyl ester with a strong base yields a $\Delta H$ of exactly $-48 \text{ to } -52\text{ kJ/mol}$. We will lock this in at $-50\text{ kJ/mol}$.

Fluid Properties ($C_p$ and $\rho$): The system uses a mixture of DMSO and 2-MeTHF. The specific heat capacity ($C_p$) of pure DMSO is $1.96\text{ J/(g}\cdot\text{K)}$ and 2-MeTHF is $\sim 1.9\text{ J/(g}\cdot\text{K)}$. The density of DMSO is $1.10\text{ g/mL}$ and 2-MeTHF is $0.85\text{ g/mL}$. A mixed process stream is therefore safely estimated at $C_p = 1.95\text{ J/(g}\cdot\text{K)}$ and $\rho = 0.95\text{ g/mL}$.

In [ ]:
import math
import pandas as pd
from datetime import datetime
from IPython.display import display, HTML

# ==================================================== #
#                 USER CONFIGURATION                   #
# ==================================================== #

# --- EXPORT SETTINGS ---
EXPORT_TO_EXCEL = True
EXCEL_PREFIX = 'Cipro_Mass_Balance'

# --- TANK MAPPING (FROM PFD TABLE) ---
TANK_MAPPING = {
    'Feed 1 (Acrylate)': 'TK-102',
    'Feed 2 (Amine)': 'TK-103',
    'Feed 3 (Piperazine)': 'TK-109',
    'Feed 4 (TBAOH)': 'TK-110',
    'T1 - Precip Agent': 'TK-208',
    'T2 - CIP Solvent (TBAOH)': 'TK-206',
    'T3 - Wash 1 (Acetone)': 'TK-205',
    'T4 - Wash 2 (Water)': 'TK-204',
    'T5 - Dissolution (HCl)': 'TK-203',
    'T6 - Isopropanol Antisolvent': 'TK-202',
    'T7 - Wash Solvent (Final)': 'TK-201'
}

# --- MODE AND TARGETS ---
CALCULATION_MODE = 'backward'
feed1_molar_flow_mol_h = 0.225
target_pure_API_kg_h = 2*6.875

# --- PROCESS YIELDS & SOLVENT RATIOS ---
YIELDS = {
    'steps_1_and_2': 0.91,
    'extraction_CLLE': 0.88,
    'DMA': 0.05,
    'steps_4_and_5': 0.90,
    'downstream_purification': 0.83
}

SOLVENT_RATIOS = {
    'T1_precip_agent_vol_ratio': 0.58,
    'T3_wash1_acetone_L_kg': 20.0,
    'T4_wash2_water_L_kg': 20.0,
    'T5_dissolution_L_kg': 16.5,
    'T6_antisolvent_vol_ratio': 1.5,
    'T7_wash_final_L_kg': 8.0
}

# --- THERMODYNAMICS & FLUID PROPERTIES ---
THERMO_PARAMS = {
    'dH_step1_2': -80.0,
    'dH_step4': -87.0,
    'dH_step5': -50.0,
    'density_kg_L': 0.95,
    'Cp_kJ_kgK': 1.95,
    'T_ambient': 25.0,
    'T_reactor_1': 150.0
}

# --- REACTOR KINETICS & FEED MOLARITIES ---
RESIDENCE_TIMES_TAU = {
    'Reactor_1_Step_1': 2.0,
    'Reactor_2_Step_2': 1.3,
    'Reactor_4_Step_4': 5.0,
    'Reactor_5_Step_5': 3.2
}

FEED_MOLARITIES = {
    'Feed_1_Acrylate': 1.0,
    'Feed_2_Amine': 1.25,
    'Feed_3_Piperazine': 2.3,
    'Feed_4_TBAOH': 1.5
}

# NEW: Feed 1 is actually the acyl chloride feed per Armstrong 2021 Scheme 4
# The acrylate is in a separate feed at 1.2 M in MeCN with 1.15 equiv DIPEA
FEED_SPEC = {
    'acyl_chloride_M': 1.0,           # mol/L in MeCN, TK-101
    'acrylate_M':      1.2,           # mol/L in MeCN, TK-102
    'DIPEA_equiv':     1.15,          # equiv relative to acrylate, co-fed in TK-102
}

# ==================================================== #
#                  CORE CALCULATION CLASSES            #
# ==================================================== #

class CiprofloxacinMassEnergyBalance:
    def __init__(self, mode, yields, ratios, thermo, target_kg_h, feed_mol_h):
        self.mode = mode
        self.yields = yields
        self.ratios = ratios
        self.thermo = thermo

        self.MW_cipro = 331.34 # g/mol
        self.MW_cipro_hcl_hydrate = 385.8 # g/mol

        self.base_feed1_mol_h = 0.225
        self.ratio_feed2 = 1.0
        self.ratio_crude_stream = 14.2 / 3.75

        self.ratio_v_dot_step1 = 7.50 / 3.75
        self.ratio_v_dot_step2 = 7.50 / 3.75
        self.ratio_v_dot_step4 = 10.85 / 3.75
        self.ratio_v_dot_step5 = 14.20 / 3.75

        self.unc = 0.088

        if self.mode == 'forward':
            self.feed1_mol_h = feed_mol_h
        elif self.mode == 'backward':
            baseline_pure_kg_h = 0.052 #from forward mode
            scale_factor = target_kg_h / baseline_pure_kg_h
            self.feed1_mol_h = self.base_feed1_mol_h * scale_factor
        else:
            raise ValueError("Mode must be 'forward' or 'backward'")

        self.feed1_flow_L_h = self.feed1_mol_h / FEED_MOLARITIES['Feed_1_Acrylate']

    def calculate(self):
        molar_1_acrylate_mol_h = self.feed1_mol_h

        feed2_flow_L_h = self.feed1_flow_L_h * self.ratio_feed2
        molar_4_amine_mol_h = feed2_flow_L_h * FEED_MOLARITIES['Feed_2_Amine']
        limiting_molar_mol_h = min(molar_1_acrylate_mol_h, molar_4_amine_mol_h)

        molar_1_acyl_chloride = limiting_molar_mol_h
        molar_3_keto_ester = limiting_molar_mol_h * 1.0
        molar_5_enamine = limiting_molar_mol_h * self.yields['steps_1_and_2']
        molar_6_dma = self.yields['DMA'] * molar_5_enamine
        molar_5_extracted = molar_5_enamine * self.yields['extraction_CLLE']
        molar_8_piperazine = molar_5_extracted
        molar_9_cipro_ester = molar_5_extracted * (self.yields['steps_4_and_5'] ** 0.5)
        molar_10_tbaoh = molar_9_cipro_ester

        molar_11_crude = molar_5_extracted * self.yields['steps_4_and_5']
        mass_crude_kg_h = (molar_11_crude * self.MW_cipro) / 1000

        molar_pure = molar_11_crude * self.yields['downstream_purification']
        mass_pure_kg_h = (molar_pure * self.MW_cipro_hcl_hydrate) / 1000

        crude_stream_vol_L_h = self.feed1_flow_L_h * self.ratio_crude_stream

        t1_precip = crude_stream_vol_L_h * self.ratios['T1_precip_agent_vol_ratio']
        t3_wash1 = mass_crude_kg_h * self.ratios['T3_wash1_acetone_L_kg']
        t4_wash2 = mass_crude_kg_h * self.ratios['T4_wash2_water_L_kg']
        t5_dissolve = mass_crude_kg_h * self.ratios['T5_dissolution_L_kg']
        t6_antisolvent = (crude_stream_vol_L_h + t1_precip) * self.ratios['T6_antisolvent_vol_ratio']
        t7_final_wash = mass_pure_kg_h * self.ratios['T7_wash_final_L_kg']

        t2_cip_solvent = molar_10_tbaoh / FEED_MOLARITIES['Feed_4_TBAOH']

        v_dot_reactors_L_h = {
            'Reactor_1_Step_1': self.feed1_flow_L_h * self.ratio_v_dot_step1,
            'Reactor_2_Step_2': self.feed1_flow_L_h * self.ratio_v_dot_step2,
            'Reactor_4_Step_4': self.feed1_flow_L_h * self.ratio_v_dot_step4,
            'Reactor_5_Step_5': self.feed1_flow_L_h * self.ratio_v_dot_step5
        }

        mol_s_step1 = limiting_molar_mol_h / 3600.0
        mol_s_step4 = molar_5_extracted / 3600.0
        mol_s_step5 = molar_9_cipro_ester / 3600.0

        q_rxn_1_2_kW = mol_s_step1 * self.thermo['dH_step1_2']
        q_rxn_4_kW = mol_s_step4 * self.thermo['dH_step4']
        q_rxn_5_kW = mol_s_step5 * self.thermo['dH_step5']
        total_reaction_heat_kW = q_rxn_1_2_kW + q_rxn_4_kW + q_rxn_5_kW

        vol_s_feed_stream_L_s = (self.feed1_flow_L_h + feed2_flow_L_h) / 3600.0
        mass_s_feed_stream_kg_s = vol_s_feed_stream_L_s * self.thermo['density_kg_L']

        dT_heat = self.thermo['T_reactor_1'] - self.thermo['T_ambient']
        q_sensible_heating_kW = mass_s_feed_stream_kg_s * self.thermo['Cp_kJ_kgK'] * dT_heat

        dT_cool = self.thermo['T_ambient'] - self.thermo['T_reactor_1']
        q_sensible_cooling_kW = mass_s_feed_stream_kg_s * self.thermo['Cp_kJ_kgK'] * dT_cool

        u = lambda val: (val, val * self.unc)

        return {
            "components": {
                "1 - Acyl Chloride": u(molar_1_acyl_chloride),
                "2 - Acrylate SM": u(molar_1_acrylate_mol_h),
                "3 - Keto-ester": u(molar_3_keto_ester),
                "4 - Cyclopropylamine": u(molar_4_amine_mol_h),
                "5 - Cyclopropyl-enamine": u(molar_5_enamine),
                "6 - Dimethylamine": u(molar_6_dma),
                "8 - Piperazine": u(molar_8_piperazine),
                "9 - Cipro-ester": u(molar_9_cipro_ester),
                "10 - TBAOH": u(molar_10_tbaoh),
                "11 - Crude Ciprofloxacin": u(molar_11_crude),
            },
            "solvents": {
                "T1 - Precip Agent": u(t1_precip),
                "T2 - CIP Solvent (TBAOH)": u(t2_cip_solvent),
                "T3 - Wash 1 (Acetone)": u(t3_wash1),
                "T4 - Wash 2 (Water)": u(t4_wash2),
                "T5 - Dissolution (HCl)": u(t5_dissolve),
                "T6 - Isopropanol Antisolvent": u(t6_antisolvent),
                "T7 - Wash Solvent (Final)": u(t7_final_wash),
            },
            "metrics": {
                "Crude API Rate (kg/h)": u(mass_crude_kg_h),
                "Pure API Rate (kg/h)": u(mass_pure_kg_h),
                "q_rxn_1_2 (kW)": q_rxn_1_2_kW,
                "q_rxn_4 (kW)": q_rxn_4_kW,
                "q_rxn_5 (kW)": q_rxn_5_kW,
                "q_rxn_total (kW)": total_reaction_heat_kW,
                "q_sensible_heat (kW)": q_sensible_heating_kW,
                "q_sensible_cool (kW)": q_sensible_cooling_kW,
            },
            "v_dots": v_dot_reactors_L_h,
            "raw_n_dots": {
                'acrylate': molar_1_acrylate_mol_h,
                'amine': molar_4_amine_mol_h,
                'piperazine': molar_8_piperazine,
                'tbaoh': molar_10_tbaoh
            }
        }

class PFRVolumeCalculator:
    def __init__(self, tau_min, v_dot_L_h):
        self.tau_min = tau_min
        self.v_dot_L_h = v_dot_L_h

    def calculate_volumes(self):
        volumes = {}
        for reactor in self.tau_min.keys():
            tau_hr = self.tau_min[reactor] / 60.0
            v_L = self.v_dot_L_h[reactor] * tau_hr
            volumes[reactor] = {'Volume (L)': v_L, 'v_dot (L/h)': self.v_dot_L_h[reactor], 'tau (hours)': tau_hr}
        return volumes

class ContinuousSpeciesTracker:
    def __init__(self, feed_molarities, req_n_dots, yields):
        self.molarities = feed_molarities
        self.yield_1_2 = yields['steps_1_and_2']
        self.DMA = yields['DMA']
        self.yield_clle = yields['extraction_CLLE']
        self.yield_4 = math.sqrt(yields['steps_4_and_5'])
        self.yield_5 = math.sqrt(yields['steps_4_and_5'])

        self.req_n_dot_acrylate = req_n_dots['acrylate']
        self.req_n_dot_amine = req_n_dots['amine']
        self.req_n_dot_piperazine = req_n_dots['piperazine']
        self.req_n_dot_tbaoh = req_n_dots['tbaoh']

        self.v_dot_feeds = {
            'Feed_1_Acrylate': self.req_n_dot_acrylate / self.molarities['Feed_1_Acrylate'],
            'Feed_2_Amine': self.req_n_dot_amine / self.molarities['Feed_2_Amine'],
            'Feed_3_Piperazine': self.req_n_dot_piperazine / self.molarities['Feed_3_Piperazine'],
            'Feed_4_TBAOH': self.req_n_dot_tbaoh / self.molarities['Feed_4_TBAOH']
        }

        self.stream = {
            '2_Acrylate': 0.0, '4_Amine': 0.0, '5_Enamine': 0.0, '6_DMA': 0.0,
            '8_Piperazine': 0.0, '9_Cipro_Ester': 0.0, '10_TBAOH': 0.0, '11_Crude_Cipro': 0.0
        }
        self.v_dot_total = 0.0
        self.report = {}

    def run_process(self):
        self.v_dot_total = self.v_dot_feeds['Feed_1_Acrylate'] + self.v_dot_feeds['Feed_2_Amine']
        n_dot_reacted = self.req_n_dot_acrylate * self.yield_1_2
        self.stream['2_Acrylate'] = self.req_n_dot_acrylate - n_dot_reacted
        self.stream['4_Amine'] = self.req_n_dot_amine - n_dot_reacted
        self.stream['5_Enamine'] = n_dot_reacted
        self.stream['6_DMA'] = self.DMA * n_dot_reacted
        self.report['1_Acylation_Enamine'] = {'v_dot': self.v_dot_total, 'stream': self.stream.copy()}

        self.stream['5_Enamine'] = self.stream['5_Enamine'] * self.yield_clle
        self.stream['6_DMA'] = 0.0
        self.stream['4_Amine'] = 0.0
        self.report['2_CLLE_Organic'] = {'v_dot': self.v_dot_total, 'stream': self.stream.copy()}

        self.v_dot_total += self.v_dot_feeds['Feed_3_Piperazine']
        self.stream['8_Piperazine'] += self.req_n_dot_piperazine
        n_dot_reacted_4 = self.stream['5_Enamine'] * self.yield_4
        self.stream['5_Enamine'] -= n_dot_reacted_4
        self.stream['8_Piperazine'] -= n_dot_reacted_4
        self.stream['9_Cipro_Ester'] += n_dot_reacted_4
        self.report['3_Ring_Closure'] = {'v_dot': self.v_dot_total, 'stream': self.stream.copy()}

        self.v_dot_total += self.v_dot_feeds['Feed_4_TBAOH']
        self.stream['10_TBAOH'] += self.req_n_dot_tbaoh
        n_dot_reacted_5 = self.stream['9_Cipro_Ester'] * self.yield_5
        self.stream['9_Cipro_Ester'] -= n_dot_reacted_5
        self.stream['10_TBAOH'] -= n_dot_reacted_5
        self.stream['11_Crude_Cipro'] += n_dot_reacted_5
        self.report['4_Hydrolysis'] = {'v_dot': self.v_dot_total, 'stream': self.stream.copy()}

        return self.report

# ==================================================== #
#                  EXECUTION & REPORTING               #
# ==================================================== #

def generate_reports():
    balance = CiprofloxacinMassEnergyBalance(CALCULATION_MODE, YIELDS, SOLVENT_RATIOS, THERMO_PARAMS, target_pure_API_kg_h, feed1_molar_flow_mol_h)
    res = balance.calculate()

    pfr_calc = PFRVolumeCalculator(RESIDENCE_TIMES_TAU, res['v_dots'])
    volumes = pfr_calc.calculate_volumes()

    tracker = ContinuousSpeciesTracker(FEED_MOLARITIES, res['raw_n_dots'], YIELDS)
    tracker_report = tracker.run_process()

    # ==================================================== #
    #                   DISPLAY FORMATTING                 #
    # ==================================================== #
    pd.options.display.float_format = '{:,.2f}'.format

    # Build all the DataFrames up front (unchanged calculations, just reorganized)
    df_comp = pd.DataFrame.from_dict(
        res['components'], orient='index',
        columns=['Molar Flow (mol/h)', 'Uncertainty (±)']
    )
    df_comp.index.name = 'Component'

    df_solv = pd.DataFrame.from_dict(
        res['solvents'], orient='index',
        columns=['Volumetric Flow (L/h)', 'Uncertainty (±)']
    )
    df_solv.index.name = 'Solvent Stream'

    daily_production_kg = res['metrics']['Pure API Rate (kg/h)'][0] * 12

    metrics = [
        ("Crude API Mass Rate",           f"{res['metrics']['Crude API Rate (kg/h)'][0]:.2f} kg/h"),
        ("Pure API Mass Rate",            f"{res['metrics']['Pure API Rate (kg/h)'][0]:.2f} kg/h"),
        ("Daily Production (12 h shift)", f"{daily_production_kg:.1f} kg/day"),
        ("Pre-Heating Feed Duty",         f"{res['metrics']['q_sensible_heat (kW)']:.2f} kW"),
        ("Quenching Duty",                f"{res['metrics']['q_sensible_cool (kW)']:.2f} kW"),
        ("Total Isothermal Rxn Load",     f"{abs(res['metrics']['q_rxn_total (kW)']):.2f} kW"),
    ]
    df_metrics = pd.DataFrame(metrics, columns=['Metric', 'Value']).set_index('Metric')

    df_vols = pd.DataFrame.from_dict(volumes, orient='index')
    df_vols.columns = ['Volume (L)', 'Flow Rate (L/h)', 'τ (h)']
    df_vols.index = df_vols.index.str.replace('_', ' ')
    df_vols.index.name = 'Reactor'

    tracker_data = {}
    for stage, data in tracker_report.items():
        row = {'v̇ total (L/h)': data['v_dot']}
        row.update({k.replace('_', ' ') + ' (mol/h)': v
                    for k, v in data['stream'].items() if v > 0.001})
        tracker_data[stage.replace('_', ' ')] = row
    df_tracker = pd.DataFrame.from_dict(tracker_data, orient='index').fillna(0)
    df_tracker.index.name = 'Process Stage'

    # ==================================================== #
    #         DOWNSTREAM BATCH VESSEL SIZING               #
    # ==================================================== #
    run_time_hours = 12
    batch_headspace_margin = 0.25

    vol_crude = res['v_dots']['Reactor_5_Step_5'] * run_time_hours
    vol_T1 = res['solvents'].get('T1 - Precip Agent', (0, 0))[0] * run_time_hours
    vol_T2 = res['solvents'].get('T2 - CIP Solvent (TBAOH)', (0, 0))[0] * run_time_hours
    vol_T3 = res['solvents'].get('T3 - Wash 1 (Acetone)', (0, 0))[0] * run_time_hours
    vol_T4 = res['solvents'].get('T4 - Wash 2 (Water)', (0, 0))[0] * run_time_hours
    vol_T5 = res['solvents'].get('T5 - Dissolution (HCl)', (0, 0))[0] * run_time_hours
    vol_T6 = res['solvents'].get('T6 - Isopropanol Antisolvent', (0, 0))[0] * run_time_hours
    vol_T7 = res['solvents'].get('T7 - Wash Solvent (Final)', (0, 0))[0] * run_time_hours

    v1_working_vol = vol_crude + vol_T1 + vol_T2
    v1_required_vol = v1_working_vol * (1 + batch_headspace_margin)
    v2_working_vol = max(vol_T3, vol_T4, vol_T5 + vol_T6, vol_T7)
    v2_required_vol = v2_working_vol * (1 + batch_headspace_margin)

    df_sizing = pd.DataFrame([
        ['Precipitation Unit (PR-201)', v1_working_vol, v1_required_vol],
        ['Workup Unit (WU-201)',        v2_working_vol, v2_required_vol],
    ], columns=['Unit', f'{run_time_hours} h Working Volume (L)', f'Required Size (+{int(batch_headspace_margin*100)}%) (L)'])
    df_sizing = df_sizing.set_index('Unit')

    # ==================================================== #
    #         CLLE SIZING (5:2:3 AQ:ORG:RXN)               #
    # ==================================================== #
    rxn_stream_v_dot = res['v_dots']['Reactor_2_Step_2']
    vol_ratio_unit = rxn_stream_v_dot / 3.0
    hcl_v_dot = vol_ratio_unit * 5.0
    methf_v_dot = vol_ratio_unit * 2.0

    extraction_v_dots = {
        'TK-106 (1M HCl)':   hcl_v_dot,
        'TK-107 (2-MeTHF)':  methf_v_dot
    }

    clle_feed_flow_L_h = rxn_stream_v_dot + hcl_v_dot + methf_v_dot
    clle_total_flow_mL_min = (clle_feed_flow_L_h * 1000) / 60.0
    clle_flow_per_unit_mL_min = clle_total_flow_mL_min / 2.0
    clle_safety_margin = 0.25
    required_separator_capacity_mL_min = clle_flow_per_unit_mL_min * (1 + clle_safety_margin)

    df_clle = pd.DataFrame([
        ['Reaction stream (from FLOW-2)',    f"{rxn_stream_v_dot:>10.2f}  L/h"],
        ['1M HCl feed (TK-106)',             f"{hcl_v_dot:>10.2f}  L/h"],
        ['2-MeTHF feed (TK-107)',            f"{methf_v_dot:>10.2f}  L/h"],
        ['Total combined CLLE feed',         f"{clle_total_flow_mL_min:>10.2f}  mL/min"],
        ['Flow per parallel unit (×2)',      f"{clle_flow_per_unit_mL_min:>10.2f}  mL/min"],
        [f'Required unit capacity (+{int(clle_safety_margin*100)}%)', f"{required_separator_capacity_mL_min:>10.2f}  mL/min"],
    ], columns=['Parameter', 'Value']).set_index('Parameter')

    # ==================================================== #
    #              STORAGE TANK SIZING                     #
    # ==================================================== #
    tank_buffer_margin = 0.10

    feed_v_dots = {
        f"{TANK_MAPPING['Feed 1 (Acrylate)']} — Acrylate":    res['raw_n_dots']['acrylate']   / FEED_MOLARITIES['Feed_1_Acrylate'],
        f"{TANK_MAPPING['Feed 2 (Amine)']} — Cyclopropylamine": res['raw_n_dots']['amine']    / FEED_MOLARITIES['Feed_2_Amine'],
        f"{TANK_MAPPING['Feed 3 (Piperazine)']} — Piperazine":  res['raw_n_dots']['piperazine'] / FEED_MOLARITIES['Feed_3_Piperazine'],
        f"{TANK_MAPPING['Feed 4 (TBAOH)']} — TBAOH":            res['raw_n_dots']['tbaoh']    / FEED_MOLARITIES['Feed_4_TBAOH'],
    }
    feed_v_dots.update(extraction_v_dots)

    solvent_v_dots = {}
    for k, v in res['solvents'].items():
        if k in TANK_MAPPING:
            tk_name = TANK_MAPPING[k]
            component_name = k.split(' - ')[1] if ' - ' in k else k
            solvent_v_dots[f"{tk_name} — {component_name}"] = v[0]

    upstream_tanks = {}
    downstream_tanks = {}
    for tank_name, v_dot in feed_v_dots.items():
        working_vol = v_dot * run_time_hours
        required_vol = working_vol * (1 + tank_buffer_margin)
        upstream_tanks[tank_name] = [v_dot, working_vol, required_vol]

    for tank_name, v_dot in solvent_v_dots.items():
        working_vol = v_dot * run_time_hours
        required_vol = working_vol * (1 + tank_buffer_margin)
        if "TK-1" in tank_name:
            upstream_tanks[tank_name] = [v_dot, working_vol, required_vol]
        else:
            downstream_tanks[tank_name] = [v_dot, working_vol, required_vol]

    tank_cols = ['Flow Rate (L/h)', f'{run_time_hours} h Working Vol (L)', f'Required Size +{int(tank_buffer_margin*100)}% (L)']
    df_upstream_tanks = pd.DataFrame.from_dict(upstream_tanks, orient='index', columns=tank_cols)
    df_upstream_tanks.index.name = 'Upstream Tank'
    df_downstream_tanks = pd.DataFrame.from_dict(downstream_tanks, orient='index', columns=tank_cols)
    df_downstream_tanks.index.name = 'Downstream Tank'

    divert_time_hours = 2
    reactir_flow = res['v_dots']['Reactor_2_Step_2']
    raman_flow = res['v_dots']['Reactor_2_Step_2']
    divert_tanks = {
        'TK-104 — ReactIR Divert':    [reactir_flow, reactir_flow * divert_time_hours, reactir_flow * divert_time_hours * (1 + tank_buffer_margin)],
        'TK-111 — Raman Spec Divert': [raman_flow,   raman_flow   * divert_time_hours, raman_flow   * divert_time_hours * (1 + tank_buffer_margin)],
    }
    df_divert_tanks = pd.DataFrame.from_dict(
        divert_tanks, orient='index',
        columns=['Process Flow (L/h)', f'{divert_time_hours} h Divert Vol (L)', f'Required Size +{int(tank_buffer_margin*100)}% (L)']
    )
    df_divert_tanks.index.name = 'Divert Tank'

    # ==================================================== #
    #                   EXCEL EXPORT                       #
    # ==================================================== #
    if EXPORT_TO_EXCEL:
        date_str = datetime.now().strftime("%m-%d-%y")
        dynamic_filename = f"{EXCEL_PREFIX}_{date_str}_{daily_production_kg:.0f}kg.xlsx"
        with pd.ExcelWriter(dynamic_filename) as writer:
            df_metrics.to_excel(writer, sheet_name='Summary')
            df_comp.to_excel(writer, sheet_name='Upstream_Flows')
            df_solv.to_excel(writer, sheet_name='Downstream_Flows')
            df_vols.to_excel(writer, sheet_name='Reactor_Volumes')
            df_tracker.to_excel(writer, sheet_name='Species_Tracker')
            df_sizing.to_excel(writer, sheet_name='Downstream_Vessels')
            df_clle.to_excel(writer, sheet_name='CLLE_Sizing')
            df_upstream_tanks.to_excel(writer, sheet_name='Upstream_Tanks')
            df_downstream_tanks.to_excel(writer, sheet_name='Downstream_Tanks')
            df_divert_tanks.to_excel(writer, sheet_name='Divert_Tanks')
        print(f"Exported to: {dynamic_filename}\n")

    # ==================================================== #
    #                   DISPLAY REPORT                     #
    # ==================================================== #
    def section(title):
        display(HTML(
            f"<h3 style='margin-top:28px; margin-bottom:8px; "
            f"padding:6px 12px; background:#f0f0f0; border-left:4px solid #333;'>"
            f"{title}</h3>"
        ))

    display(HTML(
        f"<h2 style='margin-bottom:4px;'>Ciprofloxacin Continuous Manufacturing — Mass &amp; Energy Balance</h2>"
        f"<p style='color:#666; margin-top:0;'>"
        f"Mode: <b>{CALCULATION_MODE.upper()}</b> &nbsp;|&nbsp; "
        f"Target: <b>{target_pure_API_kg_h:.2f} kg/h</b> pure API &nbsp;|&nbsp; "
        f"Daily output: <b>{daily_production_kg:.1f} kg/day</b> (12 h shift)"
        f"</p>"
    ))

    section("1. Production & Thermal Summary")
    display(df_metrics)

    section("2. Upstream Molar Flows")
    display(df_comp)

    section("3. Reactor Sizing (from residence time × flow)")
    display(df_vols)

    section("4. Continuous Species Tracker — Stage-by-Stage Molar Flows")
    display(df_tracker)

    section("5. CLLE Sizing (two parallel units, 5:2:3 aqueous:organic:reaction)")
    display(df_clle)

    section("6. Downstream Solvent Flows")
    display(df_solv)

    section("7. Downstream Batch Vessel Sizing (12 h run, +25% headspace)")
    display(df_sizing)

    section("8. Upstream Storage Tanks (12 h hold, +10% overhead)")
    display(df_upstream_tanks)

    section("9. Downstream Storage Tanks (12 h hold, +10% overhead)")
    display(df_downstream_tanks)

    section("10. PAT Divert Tanks (2 h divert window, +10% overhead)")
    display(df_divert_tanks)

if __name__ == "__main__":
    generate_reports()

Exported to: Cipro_Mass_Balance_04-13-26_165kg.xlsx



,Value
Metric,
Crude API Mass Rate,14.21 kg/h
Pure API Mass Rate,13.73 kg/h
Daily Production (12 h shift),164.8 kg/day
Pre-Heating Feed Duty,7.65 kW
Quenching Duty,-7.65 kW
Total Isothermal Rxn Load,3.10 kW


,Molar Flow (mol/h),Uncertainty (±)
Component,,
1 - Acyl Chloride,59.50,5.24
2 - Acrylate SM,59.50,5.24
3 - Keto-ester,59.50,5.24
4 - Cyclopropylamine,74.37,6.54
5 - Cyclopropyl-enamine,54.14,4.76
6 - Dimethylamine,2.71,0.24
8 - Piperazine,47.64,4.19
9 - Cipro-ester,45.20,3.98
10 - TBAOH,45.20,3.98


,Volume (L),Flow Rate (L/h),τ (h)
Reactor,,,
Reactor 1 Step 1,3.97,118.99,0.03
Reactor 2 Step 2,2.58,118.99,0.02
Reactor 4 Step 4,14.34,172.14,0.08
Reactor 5 Step 5,12.02,225.29,0.05


,v̇ total (L/h),2 Acrylate (mol/h),4 Amine (mol/h),5 Enamine (mol/h),6 DMA (mol/h),8 Piperazine (mol/h),9 Cipro Ester (mol/h),10 TBAOH (mol/h),11 Crude Cipro (mol/h)
Process Stage,,,,,,,,,
1 Acylation Enamine,118.99,5.35,20.23,54.14,2.71,0.00,0.00,0.00,0.00
2 CLLE Organic,118.99,5.35,0.00,47.64,0.00,0.00,0.00,0.00,0.00
3 Ring Closure,139.71,5.35,0.00,2.44,0.00,2.44,45.20,0.00,0.00
4 Hydrolysis,169.84,5.35,0.00,2.44,0.00,2.44,2.32,2.32,42.88


...

,Volumetric Flow (L/h),Uncertainty (±)
Solvent Stream,,
T1 - Precip Agent,130.67,11.50
T2 - CIP Solvent (TBAOH),30.13,2.65
T3 - Wash 1 (Acetone),284.15,25.01
T4 - Wash 2 (Water),284.15,25.01
T5 - Dissolution (HCl),234.43,20.63
T6 - Isopropanol Antisolvent,533.93,46.99
T7 - Wash Solvent (Final),109.84,9.67


,12 h Working Volume (L),Required Size (+25%) (L)
Unit,,
Precipitation Unit (PR-201),"4,633.06","5,791.32"
Workup Unit (WU-201),"9,220.32","11,525.40"


,Flow Rate (L/h),12 h Working Vol (L),Required Size +10% (L)
Upstream Tank,,,
TK-102 — Acrylate,59.50,713.94,785.34
TK-103 — Cyclopropylamine,59.50,713.94,785.34
TK-109 — Piperazine,20.71,248.58,273.43
TK-110 — TBAOH,30.13,361.59,397.75
TK-106 (1M HCl),198.32,"2,379.81","2,617.79"
TK-107 (2-MeTHF),79.33,951.92,"1,047.12"


,Flow Rate (L/h),12 h Working Vol (L),Required Size +10% (L)
Downstream Tank,,,
TK-208 — Precip Agent,130.67,"1,568.01","1,724.81"
TK-206 — CIP Solvent (TBAOH),30.13,361.59,397.75
TK-205 — Wash 1 (Acetone),284.15,"3,409.84","3,750.82"
TK-204 — Wash 2 (Water),284.15,"3,409.84","3,750.82"
TK-203 — Dissolution (HCl),234.43,"2,813.12","3,094.43"
TK-202 — Isopropanol Antisolvent,533.93,"6,407.20","7,047.92"
TK-201 — Wash Solvent (Final),109.84,"1,318.14","1,449.95"


,Process Flow (L/h),2 h Divert Vol (L),Required Size +10% (L)
Divert Tank,,,
TK-104 — ReactIR Divert,118.99,237.98,261.78
TK-111 — Raman Spec Divert,118.99,237.98,261.78


In [ ]:
# ========================================================== #
#      KG OF EACH RAW MATERIAL PER 1 KG CIPRO HCl OUTPUT     #
# ========================================================== #

# Run balance
balance = CiprofloxacinMassEnergyBalance(
    CALCULATION_MODE, YIELDS, SOLVENT_RATIOS, THERMO_PARAMS,
    target_pure_API_kg_h, feed1_molar_flow_mol_h
)
res = balance.calculate()

# Pure ciprofloxacin-HCl production rate (kg/h)
cipro_hcl_rate_kg_h = res['metrics']['Pure API Rate (kg/h)'][0]

# Normalize everything to per 1 kg of final product
scale_factor = 1 / cipro_hcl_rate_kg_h

# -----------------------------
# Molecular weights (kg/mol)
# -----------------------------
MW = {
    'acyl_chloride': 78.5/1000,    # UPDATE IF YOU HAVE EXACT MW
    'acrylate': 130.1/1000,        # UPDATE IF NEEDED
    'cyclopropylamine': 57.1/1000,
    'HCl': 36.46/1000,
    'piperazine': 86.14/1000,
    'TBAOH': 259.47/1000,
    'acetone': 58.08/1000,
    'MeTHF': 86.13/1000
}

# -----------------------------
# Extract required molar flows
# (from upstream model)
# -----------------------------
req = res['raw_n_dots']   # mol/h for acrylate, amine, piperazine, TBAOH

# -----------------------------
# Solvent volumetric flows (L/h)
# -----------------------------
solv = res['solvents']    # T1_precip, T3_acetone, T5_HCl, etc.
density = THERMO_PARAMS['density_kg_L']   # 0.95 kg/L

# ========================================================== #
#              Convert to kg input per 1 kg cipro            #
# ========================================================== #

kg_per_kg = {}

# --- Main reagents from molar consumption ---
kg_per_kg['Acrylate'] = req['acrylate'] * MW['acrylate'] * scale_factor
kg_per_kg['Cyclopropylamine'] = req['amine'] * MW['cyclopropylamine'] * scale_factor
kg_per_kg['Piperazine'] = req['piperazine'] * MW['piperazine'] * scale_factor
kg_per_kg['TBAOH (aq)'] = req['tbaoh'] * MW['TBAOH'] * scale_factor

# --- Acyl chloride assumed equimolar to acrylate limiting reagent ---
kg_per_kg['Acyl Chloride'] = req['acrylate'] * MW['acyl_chloride'] * scale_factor

# --- Solvents (convert L/h → kg/h → kg per kg product) ---
kg_per_kg['2-MeTHF'] = solv['T1 - Precip Agent'][0] * density * scale_factor
kg_per_kg['Acetone'] = solv['T3 - Wash 1 (Acetone)'][0] * density * scale_factor
kg_per_kg['HCl (aq)'] = solv['T5 - Dissolution (HCl)'][0] * density * scale_factor

# ========================================================== #
#                   OUTPUT TABLE                             #
# ========================================================== #
import pandas as pd

df_inputs = pd.DataFrame.from_dict(
    kg_per_kg, orient='index', columns=['kg input per kg Cipro-HCl']
)

# display(df_inputs)

#============================================================#
#============================================================#
#============================================================#
#============================================================#
#============================================================#
#============================================================#

# ========================================================== #
#         CORRECTED RAW MATERIAL COST PER 1 KG CIPRO HCl     #
# ========================================================== #

import pandas as pd

# -----------------------------
# Molecular weights (kg/mol)
# -----------------------------
MW = {
    'acyl_chloride': 78.5/1000,
    'acrylate': 130.1/1000,
    'cyclopropylamine': 57.1/1000,
    'HCl': 36.46/1000,
    'piperazine': 86.14/1000,
    'TBAOH': 259.47/1000,
    'acetone': 58.08/1000,
    'MeTHF': 86.13/1000,
    'MeCN': 41.05/1000,        # NEW
    'DIPEA': 129.25/1000,      # NEW
}

# -----------------------------
# Reagent densities (kg/L) — needed to back out solvent volume from feed molarity
# -----------------------------
RHO_REAGENT = {
    'acyl_chloride': 1.55,    # 2,4,5-trifluorobenzoyl chloride ~1.55 g/mL
    'acrylate':      1.01,    # ethyl 3-(dimethylamino)acrylate ~1.01 g/mL
    'MeCN':          0.786,
    'DIPEA':         0.742,
}

# -----------------------------
# Prices ($/kg)
# -----------------------------
PRICES = {
    'acyl_chloride':      2.00,
    'acrylate':           1.40,
    'cyclopropylamine':   700.0,   # revisit at bulk scale
    'piperazine':         30.0,
    'TBAOH_pure':         915.0,
    'acetonitrile':       3.0,     # T-1 component
    'acetic_acid':        0.60,    # T-1 component
    'acetone':            2.60,    # T-3
    'HCl_aq':             2.50,    # T-5 component
    'formic_acid':        650.0,   # T-5 component — CHECK YOUR SUPPLIER
    'isopropanol':        1.20,    # T-6
    'water':              0.001,   # negligible but included
    'DIPEA':              25.0,    # NEW — industrial grade N,N-diisopropylethylamine
}

# -----------------------------
# Densities (kg/L)
# -----------------------------
# Per your THERMO_PARAMS for generic streams,
# but we override per-stream below with mixture estimates
RHO = {
    'T1':  0.87,   # 50% MeCN (0.786) + 42% water (1.0) + 8% AcOH (1.049) — vol-weighted
    'T3':  0.791,  # pure acetone
    'T5':  1.01,   # dilute aqueous HCl + formic acid mixture
    'T6':  0.786,  # isopropanol
}

# -----------------------------
# T-1: Precipitation Agent
# Composition: 50 vol% MeCN, 42 vol% water, 8 vol% acetic acid
# -----------------------------
T1_vol_L_h = solv['T1 - Precip Agent'][0]          # total L/h from your model
T1_kg_h    = T1_vol_L_h * RHO['T1']                # total kg/h of mixture

# Mass fractions from vol% × density / mixture density
rho_MeCN  = 0.786;  rho_water = 1.000;  rho_AcOH = 1.049
rho_T1    = RHO['T1']

wf_MeCN  = (0.50 * rho_MeCN)  / rho_T1
wf_water = (0.42 * rho_water) / rho_T1
wf_AcOH  = (0.08 * rho_AcOH)  / rho_T1

T1_cost_per_kg_h = (
    wf_MeCN  * PRICES['acetonitrile'] +
    wf_water * PRICES['water']        +
    wf_AcOH  * PRICES['acetic_acid']
)  # $/kg mixture

# -----------------------------
# T-3: Wash 1 — pure acetone (correct as-is)
# -----------------------------
T3_vol_L_h = solv['T3 - Wash 1 (Acetone)'][0]
T3_kg_h    = T3_vol_L_h * RHO['T3']

# -----------------------------
# T-5: Dissolution Solvent
# Composition: 0.35M HCl in 25 vol% formic acid, 75 vol% water
# Consumption basis: 16.5 mL solvent / g API  ← use this directly
# -----------------------------
# 16.5 mL/g API = 16.5 L/kg API
T5_L_per_kg_cipro = 16.5                            # directly from PFD legend
T5_kg_per_kg_cipro = T5_L_per_kg_cipro * RHO['T5']

rho_FA    = 1.220   # formic acid density kg/L
rho_T5    = RHO['T5']

wf_FA_T5    = (0.25 * rho_FA)    / rho_T5
wf_water_T5 = (0.75 * rho_water) / rho_T5

# HCl contribution: 0.35 mol/L × 0.03646 kg/mol = 0.01276 kg/L → small but included
HCl_kg_per_L_T5 = 0.35 * MW['HCl']   # kg HCl per L of T-5 solution

T5_cost_per_kg = (
    wf_FA_T5    * PRICES['formic_acid'] +
    wf_water_T5 * PRICES['water']       +
    HCl_kg_per_L_T5 / rho_T5 * PRICES['HCl_aq']
)  # $/kg mixture

# -----------------------------
# T-6: Isopropanol Antisolvent
# -----------------------------
T6_vol_L_h = solv.get('T6 - Isopropanol Antisolvent', [0])[0]
T6_kg_h    = T6_vol_L_h * RHO['T6']

# -----------------------------
# T-2: TBAOH reagent
# Composition: 0.75 M aqueous TBAOH
# MW TBAOH = 259.47 g/mol → 0.75 mol/L × 0.25947 kg/mol = 0.1946 kg TBAOH per L
# -----------------------------
TBAOH_kg_per_L = 0.75 * MW['TBAOH']                 # = 0.1946 kg pure TBAOH / L solution
T2_vol_L_h     = solv.get('T2 - CIP Solvent (TBAOH)', [0])[0]

# If TBAOH is in req (molar flow), use that — more accurate
# pure TBAOH consumed (kg/h):
TBAOH_pure_kg_h = req['tbaoh'] * MW['TBAOH']        # mol/h × kg/mol

# ========================================================== #
#         ASSEMBLE kg INPUT per 1 kg CIPRO-HCl              #
# ========================================================== #
kg_per_kg = {}

# --- Stoichiometric reagents ---
kg_per_kg['Acyl Chloride']      = req['acrylate'] * MW['acyl_chloride']    * scale_factor
kg_per_kg['Acrylate']           = req['acrylate'] * MW['acrylate']         * scale_factor
kg_per_kg['Cyclopropylamine']   = req['amine']    * MW['cyclopropylamine'] * scale_factor
kg_per_kg['Piperazine']         = req['piperazine'] * MW['piperazine']     * scale_factor
kg_per_kg['TBAOH (pure basis)'] = TBAOH_pure_kg_h                          * scale_factor

# --- NEW: Upstream acetonitrile solvent (TK-101 and TK-102 feed dilution) ---
# Per Armstrong 2021: acyl chloride is fed at 1.0 M in MeCN (TK-101),
# acrylate is fed at 1.2 M in MeCN with 1.15 equiv DIPEA (TK-102).
# MeCN solvent volume = total feed solution volume - reagent volume - DIPEA volume.

# TK-101: acyl chloride in MeCN
TK101_total_L_h       = req['acrylate'] / FEED_SPEC['acyl_chloride_M']
acyl_chloride_kg_h    = req['acrylate'] * MW['acyl_chloride']
acyl_chloride_vol_L_h = acyl_chloride_kg_h / RHO_REAGENT['acyl_chloride']
MeCN_TK101_L_h        = TK101_total_L_h - acyl_chloride_vol_L_h

# TK-102: acrylate + DIPEA in MeCN
TK102_total_L_h       = req['acrylate'] / FEED_SPEC['acrylate_M']
acrylate_kg_h         = req['acrylate'] * MW['acrylate']
acrylate_vol_L_h      = acrylate_kg_h / RHO_REAGENT['acrylate']
DIPEA_mol_h           = req['acrylate'] * FEED_SPEC['DIPEA_equiv']
DIPEA_kg_h            = DIPEA_mol_h * MW['DIPEA']
DIPEA_vol_L_h         = DIPEA_kg_h / RHO_REAGENT['DIPEA']
MeCN_TK102_L_h        = TK102_total_L_h - acrylate_vol_L_h - DIPEA_vol_L_h

# Total upstream MeCN
MeCN_upstream_L_h     = MeCN_TK101_L_h + MeCN_TK102_L_h
MeCN_upstream_kg_h    = MeCN_upstream_L_h * RHO_REAGENT['MeCN']

kg_per_kg['MeCN (TK-101/102 upstream)'] = MeCN_upstream_kg_h * scale_factor
kg_per_kg['DIPEA (TK-102)']             = DIPEA_kg_h         * scale_factor

# --- T-1 Precipitation agent (MeCN + water + AcOH) ---
kg_per_kg['MeCN (T-1)']         = T1_kg_h * wf_MeCN  * scale_factor
kg_per_kg['Acetic Acid (T-1)']  = T1_kg_h * wf_AcOH  * scale_factor

# --- T-3 Acetone wash ---
kg_per_kg['Acetone (T-3)']      = T3_kg_h * scale_factor

# --- T-5 Dissolution ---
kg_per_kg['Formic Acid (T-5)']  = T5_kg_per_kg_cipro * wf_FA_T5
kg_per_kg['HCl (T-5)']          = T5_L_per_kg_cipro  * HCl_kg_per_L_T5

# --- T-6 Isopropanol antisolvent ---
kg_per_kg['Isopropanol (T-6)']  = T6_kg_h * scale_factor

# ========================================================== #
#                   COST TABLE                               #
# ========================================================== #

price_map = {
    'Acyl Chloride':             PRICES['acyl_chloride'],
    'Acrylate':                  PRICES['acrylate'],
    'Cyclopropylamine':          PRICES['cyclopropylamine'],
    'Piperazine':                PRICES['piperazine'],
    'TBAOH (pure basis)':        PRICES['TBAOH_pure'],
    'MeCN (TK-101/102 upstream)': PRICES['acetonitrile'],  # NEW
    'DIPEA (TK-102)':            PRICES.get('DIPEA', 25.0), # NEW — add DIPEA to PRICES dict
    'MeCN (T-1)':                PRICES['acetonitrile'],
    'Acetic Acid (T-1)':         PRICES['acetic_acid'],
    'Acetone (T-3)':             PRICES['acetone'],
    'Formic Acid (T-5)':         PRICES['formic_acid'],
    'HCl (T-5)':                 PRICES['HCl_aq'],
    'Isopropanol (T-6)':         PRICES['isopropanol'],
}

rows = []
total = 0
for chemical, kg_val in kg_per_kg.items():
    price = price_map.get(chemical, 0)
    cost  = kg_val * price
    total += cost
    rows.append({
        'Chemical':                    chemical,
        'kg per kg Cipro-HCl':         round(kg_val, 3),
        'Price ($/kg)':                price,
        'Cost ($/kg Cipro-HCl)':       round(cost, 2),
    })

rows.append({
    'Chemical':                'TOTAL',
    'kg per kg Cipro-HCl':     '',
    'Price ($/kg)':            '',
    'Cost ($/kg Cipro-HCl)':   round(total, 2),
})

df_costs = pd.DataFrame(rows).set_index('Chemical')
display(df_costs)

,kg per kg Cipro-HCl,Price ($/kg),Cost ($/kg Cipro-HCl)
Chemical,,,
Acyl Chloride,0.34,2.00,0.68
Acrylate,0.56,1.40,0.79
Cyclopropylamine,0.31,700.00,216.49
Piperazine,0.30,30.00,8.97
TBAOH (pure basis),0.85,915.00,781.53
MeCN (TK-101/102 upstream),4.95,3.00,14.85
DIPEA (TK-102),0.64,25.00,16.10
MeCN (T-1),3.74,3.00,11.22
Acetic Acid (T-1),0.80,0.60,0.48
